# HW9 — свежий Qwen, structured, TF-IDF и 10 shuffle
Выберите **GPU Tesla T4**, Python **3.11 или 3.12** (как в HW8; использовался Colab runtime 2026.07 / Python 3.12.13). Если текущий runtime использует 3.13+, сначала выберите совместимую past runtime version.
Загрузки: `hw9_colab_bundle.zip` и исходный `FAR-Trans.zip`. Старые Qwen-векторы в bundle служат только проверенным референсом: в режиме `--fresh-qwen` они не читаются для рекомендаций.
Запустите ячейки 1–2, перезапустите сеанс Python через меню, затем выполните 3–8 по порядку. Все результаты — development, на том же срезе 100 пользователей. Самостоятельный расчёт для отчёта — ячейка 7.

In [ ]:
# 1. Загрузка кода и проверенного референса HW8
import sys, zipfile, json, hashlib
from pathlib import Path
from google.colab import files
assert sys.version_info[:2] in ((3, 11), (3, 12)), 'Выберите Python 3.11/3.12 до установки пакетов'
uploaded = files.upload()  # выберите hw9_colab_bundle.zip
assert 'hw9_colab_bundle.zip' in uploaded
ROOT = Path('/content/hw9_project')
assert not ROOT.exists(), 'Для повторной подготовки используйте чистый runtime'
ROOT.mkdir()
with zipfile.ZipFile('hw9_colab_bundle.zip') as z:
    assert z.testzip() is None
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(ROOT)
provenance = json.loads((ROOT / 'hw9/bundle_manifest.json').read_text())
for name, expected in provenance['file_sha256'].items():
    assert hashlib.sha256((ROOT / name).read_bytes()).hexdigest() == expected, name
print('Bundle проверен. HEAD:', provenance['git_commit'])
del uploaded


In [ ]:
# 2. Установка pinned окружения HW8 + TF-IDF
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', '/content/hw9_project/hw9/requirements-colab.txt'], check=True)
print('Теперь: Runtime / Restart session (перезапустить Python). Затем запускайте с ячейки 3, не с ячейки 1.')


In [ ]:
# 3. Проверка после перезапуска Python
import sys, os, json, subprocess, importlib.metadata as metadata
from pathlib import Path
ROOT = Path('/content/hw9_project')
assert sys.version_info[:2] in ((3, 11), (3, 12))
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import torch
assert torch.cuda.is_available(), 'Выберите GPU T4'
assert 'T4' in torch.cuda.get_device_name(0), torch.cuda.get_device_name(0)
expected = {'numpy':'2.2.6', 'torch':'2.7.1', 'torchvision':'0.22.1', 'torchaudio':'2.7.1', 'transformers':'4.51.3', 'tokenizers':'0.21.1', 'huggingface-hub':'0.31.4', 'safetensors':'0.5.3', 'scipy':'1.17.0', 'scikit-learn':'1.8.0', 'joblib':'1.6.0', 'threadpoolctl':'3.8.0', 'cloudpickle':'3.1.3'}
for package, wanted in expected.items():
    actual = metadata.version(package)
    assert actual.split('+')[0] == wanted, (package, actual, wanted)
print(sys.version, torch.cuda.get_device_name(0), expected, sep='\n')
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'hw9', '-p', 'test_local.py', '-v'], cwd=ROOT, check=True)


In [ ]:
# 4. Исходный FAR-Trans.zip (тот же, что в HW8)
from google.colab import files
import zipfile
uploaded = files.upload()
assert 'FAR-Trans.zip' in uploaded
DATA_ROOT = ROOT / 'data/raw/uploaded'
DATA_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile('FAR-Trans.zip') as z:
    assert z.testzip() is None
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(DATA_ROOT)
candidates = [p.parent for p in DATA_ROOT.rglob('transactions.csv') if (p.parent / 'asset_information.csv').exists()]
assert len(candidates) == 1, candidates
DATA = candidates[0]
print('CSV:', DATA)
del uploaded


In [ ]:
# 5. Один полный свежий эксперимент. Не перезапускайте эту ячейку для продолжения.
from datetime import datetime, timezone
OUT = ROOT / 'hw9/outputs' / ('fresh_t4_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S'))
LOG = OUT.with_suffix('.log')
command = [sys.executable, '-u', 'hw9/run.py', '--fresh-qwen', '--data-dir', str(DATA), '--output-dir', str(OUT)]
print(' '.join(command))
with LOG.open('w', encoding='utf-8') as log:
    process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    assert process.wait() == 0, 'Ошибка запуска: см. лог ' + str(LOG)
manifest = json.loads((OUT / 'manifest.json').read_text())
assert manifest['status'] == 'completed' and manifest['mode'] == 'fresh_qwen_cuda'
print('Готово:', OUT)
print('Qwen failures:', len(manifest['qwen_fresh_inference']['failures']))


In [ ]:
# 6. Таблица двух средних, десять seed, диагностика и время
subprocess.run([sys.executable, 'hw9/summarize.py', '--output-dir', str(OUT)], cwd=ROOT, check=True)
import csv
with (OUT / 'comparison.csv').open() as f:
    for row in csv.DictReader(f):
        print(row['method'], 'vector construction:', row['encoding_seconds'], 's; recommendation:', row['scoring_seconds'], 's')
print('Shared preparation:', manifest['shared_preparation_seconds'], 's')
print('Tokenizer load:', manifest['tokenizer_load_seconds'], 's; profiles:', manifest['profile_construction_seconds'], 's')
print('Qwen model load:', manifest['model_load_seconds'], 's; inference:', manifest['qwen_inference_seconds'], 's')
print('TF-IDF dense conversion:', json.loads((OUT / 'tfidf_knn/representation.json').read_text())['densification_seconds'], 's')


In [ ]:
# 7. Материалы для ВАШЕЙ личной проверки — без подстановки старых рангов HW8
UID = '870E814DB274CFA95600'
CHECK = OUT / 'personal_check' / UID
for name in ('june_buy_source_rows.csv', 'unique_relevant_isins.json', 'qwen_knn_top10.csv', 'tfidf_knn_top10.csv'):
    print('\n' + name + '\n' + (CHECK / name).read_text(encoding='utf-8'))
print('Найдите ранги июньских ISIN в новых Top-10; повторы покупок дают один relevant ISIN.')
print('DCG = сумма 1/log2(rank+1) по попаданиям до ранга 10; IDCG = сумма 1/log2(r+1) для r=1..min(10, число уникальных relevant).')
print('После самостоятельного расчёта сравните с qwen_knn_per_user.csv и tfidf_knn_per_user.csv в той же папке.')
print('Этот вывод не является подтверждением личной проверки: её фиксирует автор после расчёта.')


In [ ]:
# 8. Финальный архив со всеми результатами и скачивание
import shutil, hashlib, zipfile
from google.colab import files
shutil.copy2(LOG, OUT / 'run.log')
shutil.copy2(ROOT / 'hw9/notebooks/hw9_fresh_qwen_t4.ipynb', OUT / 'code/hw9/hw9_fresh_qwen_t4.ipynb')
archive = Path(shutil.make_archive(str(OUT), 'zip', root_dir=OUT))
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None
print(archive.name, archive.stat().st_size, 'bytes')
print('SHA256:', hashlib.sha256(archive.read_bytes()).hexdigest())
files.download(str(archive))
